# Notebook 20 — Physical Contact Null Check

## The robustness question

Nb06 found that the Brittin/Zhen nerve-ring contact matrix predicts **static adult synaptic edges** at AUC 0.78. Nb07 found that chemical topology predicts **edge arrival** at AUC 0.76. These numbers are suspiciously close. **Could Nb07's topology signal be nothing more than a proxy for whether two neurons physically touch?**

If contact alone predicts arrival at AUC ≥ 0.74, then Nb07's "topological rule" claim is substantially weakened — topology would be a downstream consequence of 3D anatomy, not an independent rule.

## What we're doing

For each developmental transition, ask three questions:
1. **Contact-only AUC**: Using only the adult nerve-ring contact matrix, predict edge arrival at each transition.
2. **Topology vs contact head-to-head**: Which predicts better?
3. **Does topology ADD above contact?** This is the load-bearing test.

## Preregistered criteria

1. **Contact matrix is loadable** for ≥ 170 of the 185 common neurons.
2. **Contact-only AUC is in [0.55, 0.85]**. (If below 0.55, contact isn't a predictor at all; if above 0.85, it's doing all the work.)
3. **Topology-only reproduces Nb07**: AUC ∈ [0.74, 0.78].
4. **KEY TEST**: Topology + contact AUC ≥ contact-only AUC + 0.02. (Topology adds beyond 3D proximity.)
5. **Permutation null** (shuffle topology features, keep contact): shuffled delta < 0.

## Halting rule

**If criterion 4 fails** (topology doesn't add above contact): Nb07's topological rule claim weakens dramatically. Paper must reframe as "contact-mediated wiring" rather than "topological rule."

**If criterion 4 passes**: Nb07/09 are robust to the obvious confound. Topology captures something beyond physical proximity.

## Important limitation

We only have adult + L4 contact matrices (Brittin nerve-ring data). We use **adult contact as a proxy for all transitions**. This is conservative: if contact at adult stage already captures edge arrival at earlier stages, that would be the strongest possible confound claim. If even using adult contact as a proxy, topology still adds, the topology signal is robust.

In [1]:
import sys, time
from pathlib import Path
import warnings; warnings.simplefilter('ignore')
_HERE = Path.cwd()
if (_HERE / 'lib').is_dir(): sys.path.insert(0, str(_HERE))
elif (_HERE.parent / 'lib').is_dir(): sys.path.insert(0, str(_HERE.parent))

from lib.paths import DATA, DERIVED
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RNG = np.random.default_rng(42)

# Load Nb07 pooled candidates (has all topology features + transition labels)
cand = pd.read_csv(DERIVED / 'nb07_pooled_candidates.csv')
print(f'Nb07 candidates: {len(cand)}')
print(cand['transition'].value_counts().to_string())

# Common-185 neuron list
STAGES = ['L1_1','L1_2','L1_3','L1_4','L2','L3','adult']
s_neurons = {}
for s in STAGES:
    d = np.load(DERIVED / 'developmental' / f'connectome_{s}.npz', allow_pickle=True)
    s_neurons[s] = set(str(n) for n in d['neurons'])
common = sorted(set.intersection(*s_neurons.values()))
print(f'Common-185: {len(common)}')

Nb07 candidates: 98718
transition
L1_consensus->L2    33344
L3->adult           32700
L2->L3              32674
Common-185: 185


## Step 1 — Load Brittin adult nerve-ring contact matrix

In [2]:
contact_xlsx = DATA / 'connectome/nerve_ring_neighbors/Adult and L4 nerve ring neighbors.xlsx'
contact_df = pd.read_excel(contact_xlsx, sheet_name='adult nerve ring neighbors', header=0, index_col=0)
contact_df = contact_df.loc[:, ~contact_df.columns.str.startswith('Unnamed')]
contact_df = contact_df[contact_df.index.map(lambda x: isinstance(x, str))]
contact_df = contact_df.apply(pd.to_numeric, errors='coerce').fillna(0.0)

contact_neurons = set(contact_df.index)
common_with_contact = [n for n in common if n in contact_neurons]
print(f'Common-185 neurons in contact matrix: {len(common_with_contact)}/{len(common)}')

# Restrict contact matrix to common-185 and check dimensions
cm_idx = [i for i, n in enumerate(common) if n in contact_neurons]
contact_in_common = np.zeros((len(common), len(common)), dtype=np.float32)
name_to_idx = {n: i for i, n in enumerate(common)}
for row_name in contact_df.index:
    if row_name not in name_to_idx: continue
    ri = name_to_idx[row_name]
    for col_name in contact_df.columns:
        if col_name not in name_to_idx: continue
        ci = name_to_idx[col_name]
        val = contact_df.loc[row_name, col_name]
        if pd.notna(val):
            contact_in_common[ri, ci] = float(val)

# Contact is symmetric (physical adjacency); ensure
contact_in_common = (contact_in_common + contact_in_common.T) / 2
print(f'Contact matrix on common-185: {contact_in_common.shape}')
print(f'Nonzero contact entries: {int((contact_in_common > 0).sum())}')
print(f'Contact value range: [{contact_in_common.min():.0f}, {contact_in_common.max():.0f}]')

Common-185 neurons in contact matrix: 161/185


Contact matrix on common-185: (185, 185)
Nonzero contact entries: 8330
Contact value range: [0, 55661]


## Step 2 — Build contact features per candidate

In [3]:
# For each candidate (i, j) in Nb07 pool:
#   contact_area     : raw contact area value between i and j in nerve-ring EM
#   log_contact      : log1p of contact area (handles orders of magnitude)
#   contact_present  : binary — is contact > 0?

contact_area = np.array([contact_in_common[int(r['i']), int(r['j'])] for _, r in cand.iterrows()])
log_contact = np.log1p(contact_area)
contact_present = (contact_area > 0).astype(int)

print(f'Candidates with any contact: {int((contact_area > 0).sum())} / {len(cand)}')
print(f'Log-contact range: [{log_contact.min():.2f}, {log_contact.max():.2f}]')

# Sanity: among candidates with contact, what's the arrival rate vs without?
y = cand['arrived'].values.astype(int)
p_arrive_contact = (y[contact_present == 1].mean()) if (contact_present == 1).sum() else 0
p_arrive_nocontact = (y[contact_present == 0].mean()) if (contact_present == 0).sum() else 0
print(f'\nP(arrive | has contact):   {p_arrive_contact:.4f}  (n={int((contact_present == 1).sum())})')
print(f'P(arrive | no contact):    {p_arrive_nocontact:.4f}  (n={int((contact_present == 0).sum())})')
print(f'Enrichment ratio: {p_arrive_contact / max(p_arrive_nocontact, 1e-6):.2f}x')

Candidates with any contact: 22090 / 98718
Log-contact range: [0.00, 10.93]

P(arrive | has contact):   0.0681  (n=22090)
P(arrive | no contact):    0.0043  (n=76628)
Enrichment ratio: 15.67x


## Step 3 — Nested AUC comparison

In [4]:
TOP_COVS = ['out_deg_i','in_deg_j','shared_out','shared_in','triangle_closure','reverse_2step']
X_topo = cand[TOP_COVS].values.astype(float)
X_contact = np.stack([log_contact, contact_present], axis=1)
X_both = np.concatenate([X_topo, X_contact], axis=1)

def cv_auc(X, y, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    aucs = []
    for tr, te in skf.split(X, y):
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=1.0))
        m.fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return np.array(aucs)

aucs_topo = cv_auc(X_topo, y)
aucs_contact = cv_auc(X_contact, y)
aucs_both = cv_auc(X_both, y)

delta_topo_above_contact = aucs_both.mean() - aucs_contact.mean()
delta_contact_above_topo = aucs_both.mean() - aucs_topo.mean()

print(f'Topology-only AUC:      {aucs_topo.mean():.3f} ± {aucs_topo.std():.3f}')
print(f'Contact-only AUC:       {aucs_contact.mean():.3f} ± {aucs_contact.std():.3f}')
print(f'Topology + contact AUC: {aucs_both.mean():.3f} ± {aucs_both.std():.3f}')
print(f'\nDelta (topology+contact - contact-only): {delta_topo_above_contact:+.4f}')
print(f'Delta (topology+contact - topology-only): {delta_contact_above_topo:+.4f}')
print(f'\n>> Interpretation:')
print(f'   If topology AUC > contact AUC: topology captures something contact does not')
print(f'   If they are similar but topology+contact >> each alone: they capture complementary info')
print(f'   If contact alone matches topology: Nb07 signal is mostly 3D-proximity')

Topology-only AUC:      0.758 ± 0.011
Contact-only AUC:       0.836 ± 0.009
Topology + contact AUC: 0.875 ± 0.007

Delta (topology+contact - contact-only): +0.0384
Delta (topology+contact - topology-only): +0.1167

>> Interpretation:
   If topology AUC > contact AUC: topology captures something contact does not
   If they are similar but topology+contact >> each alone: they capture complementary info
   If contact alone matches topology: Nb07 signal is mostly 3D-proximity


## Step 4 — Permutation null

In [5]:
N_PERM = 100
null_delta_topo = []
t0 = time.time()
for i in range(N_PERM):
    perm = RNG.permutation(X_both.shape[0])
    X_shuf = X_both.copy()
    X_shuf[:, :len(TOP_COVS)] = X_both[perm, :len(TOP_COVS)]  # shuffle topology features; keep contact
    aucs_s = cv_auc(X_shuf, y, random_state=42+i).mean()
    null_delta_topo.append(aucs_s - aucs_contact.mean())
    if (i+1) % 25 == 0:
        print(f'  {i+1}/{N_PERM} ({time.time()-t0:.1f}s)')
null_delta_topo = np.array(null_delta_topo)
print(f'\nNull delta (topology-shuffle, keep contact): mean={null_delta_topo.mean():+.4f}, 95pct={np.percentile(null_delta_topo, 95):+.4f}')
print(f'Observed topology-above-contact delta: {delta_topo_above_contact:+.4f}')

  25/100 (30.5s)


  50/100 (62.7s)


  75/100 (94.9s)


  100/100 (125.9s)

Null delta (topology-shuffle, keep contact): mean=-0.0003, 95pct=+0.0039
Observed topology-above-contact delta: +0.0384


## Step 5 — Preregistered criteria

In [6]:
crit1 = len(common_with_contact) >= 170
crit2 = 0.55 <= aucs_contact.mean() <= 0.85
crit3 = 0.74 <= aucs_topo.mean() <= 0.78
crit4 = delta_topo_above_contact >= 0.02
crit5 = delta_topo_above_contact > np.percentile(null_delta_topo, 95)

print('PREREGISTERED CRITERIA')
print('=' * 70)
print(f'  [{"PASS" if crit1 else "FAIL"}] 1 contact matrix covers >= 170 neurons        {len(common_with_contact)}/185')
print(f'  [{"PASS" if crit2 else "FAIL"}] 2 Contact-only AUC in [0.55, 0.85]             {aucs_contact.mean():.3f}')
print(f'  [{"PASS" if crit3 else "FAIL"}] 3 Topology-only AUC in [0.74, 0.78]             {aucs_topo.mean():.3f}')
print(f'  [{"PASS" if crit4 else "FAIL"}] 4 Topology adds >= 0.02 above contact           {delta_topo_above_contact:+.4f}')
print(f'  [{"PASS" if crit5 else "FAIL"}] 5 Delta > null 95pct                            {delta_topo_above_contact:+.4f} vs {np.percentile(null_delta_topo, 95):+.4f}')
print('=' * 70)

# CRITICAL INTERPRETATION
if crit4 and crit5:
    verdict = 'POSITIVE — topology contains genuine signal BEYOND physical contact'
    implication = 'Nb07/09 topological rule is robust to the contact confound'
elif crit4 or crit5:
    verdict = 'PARTIAL — topology adds some signal above contact but weakly'
    implication = 'Topology is PARTIALLY a proxy for 3D contact; paper should acknowledge this'
else:
    verdict = 'NULL — topology does NOT add above contact'
    implication = 'CRITICAL: Nb07/09 signal is substantially a proxy for physical adjacency. Reframe paper.'

print(f'\nVERDICT: {verdict}')
print(f'IMPLICATION: {implication}')

summary = pd.DataFrame([{
    'n_common_with_contact': len(common_with_contact),
    'auc_topology_only': float(aucs_topo.mean()),
    'auc_contact_only': float(aucs_contact.mean()),
    'auc_topology_plus_contact': float(aucs_both.mean()),
    'delta_topology_above_contact': float(delta_topo_above_contact),
    'delta_contact_above_topology': float(delta_contact_above_topo),
    'null_95pct_topology_delta': float(np.percentile(null_delta_topo, 95)),
    'verdict': verdict,
    'paper_implication': implication,
}])
summary.to_csv(DERIVED / 'nb20_final_summary.csv', index=False)
print(summary.T.to_string())

PREREGISTERED CRITERIA
  [FAIL] 1 contact matrix covers >= 170 neurons        161/185
  [PASS] 2 Contact-only AUC in [0.55, 0.85]             0.836
  [PASS] 3 Topology-only AUC in [0.74, 0.78]             0.758
  [PASS] 4 Topology adds >= 0.02 above contact           +0.0384
  [PASS] 5 Delta > null 95pct                            +0.0384 vs +0.0039

VERDICT: POSITIVE — topology contains genuine signal BEYOND physical contact
IMPLICATION: Nb07/09 topological rule is robust to the contact confound
                                                                                                0
n_common_with_contact                                                                         161
auc_topology_only                                                                        0.757929
auc_contact_only                                                                         0.836153
auc_topology_plus_contact                                                                0.874584
delta_to